# 0. Helper functions

In [2]:
import os, sys
from pathlib import Path

PROJECT = Path.cwd().parent            # notebook folder → its parent = project root (where src/ and data/ are)
os.chdir(PROJECT)                      # so data/... paths in src/data.py work
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))   # so "import src" works

print("working dir:", Path.cwd())  

from src.data import binary_label, study_table, top_k_classes, usable_findings
from src.embeddings import EMBEDDINGS_DIR, embedding_path, load_embeddings

working dir: c:\Users\lucas\Desktop\4th_semester\02517_responsible_ai\code\mandatory\project3\responsible-ai-rexgradient-main


c:\Users\lucas\Desktop\4th_semester\02517_responsible_ai\code\mandatory\project3\responsible-ai-rexgradient-main\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import inspect
from src.data import study_table, findings_texts, usable_findings, binary_label, label_counts, top_k_classes
from src.embeddings import embedding_path, load_embeddings, load_encoder, embed_texts
from src.edits import sentences, find_sentence

In [6]:
st = study_table()
print(f"study_table() has {len(st)} rows, {len(st.columns)} columns")
print(st.columns)

study_table() has 10000 rows, 47 columns
Index(['study_id', 'patient_id', 'accession_number', 'study_uid', 'series_uid',
       'image_id', 'image_path', 'n_images_in_study', 'sex', 'age_years',
       'age_raw', 'ethnic_group', 'weight', 'height', 'institution',
       'manufacturer', 'manufacturer_raw', 'study_date', 'study_year',
       'study_description', 'modality', 'body_part', 'view_position',
       'view_position_raw', 'image_height', 'image_width', 'indication',
       'comparison', 'findings', 'impression',
       'chexbert_enlarged_cardiomediastinum', 'chexbert_cardiomegaly',
       'chexbert_lung_opacity', 'chexbert_lung_lesion', 'chexbert_edema',
       'chexbert_consolidation', 'chexbert_pneumonia', 'chexbert_atelectasis',
       'chexbert_pneumothorax', 'chexbert_pleural_effusion',
       'chexbert_pleural_other', 'chexbert_fracture',
       'chexbert_support_devices', 'chexbert_no_finding', 'health_system',
       'findings_words', 'impression_words'],
      dtype='ob

In [9]:
X_all = load_embeddings(embedding_path("findings"), st)
print(f"X_all has shape {X_all.shape}, dtype {X_all.dtype}")
print(f'Example embedding vector (first 5 values): {X_all[0][:5]}')

X_all has shape (10000, 768), dtype float32
Example embedding vector (first 5 values): [ 0.1829884  -0.4405663   0.09515186  0.45542675  0.20336443]


In [13]:
label = binary_label(st, "pleural_effusion")
print(label)
# print all trues and negatives
print("True labels (pleural_effusion):")
print(label[label == True])
print("False labels (pleural_effusion):")
print(label[label == False])


0       NaN
1       NaN
2       NaN
3       NaN
4       NaN
       ... 
9995    NaN
9996    NaN
9997    NaN
9998    1.0
9999    NaN
Name: chexbert_pleural_effusion, Length: 10000, dtype: float64
True labels (pleural_effusion):
13      1.0
56      1.0
59      1.0
61      1.0
115     1.0
       ... 
9947    1.0
9951    1.0
9973    1.0
9989    1.0
9998    1.0
Name: chexbert_pleural_effusion, Length: 646, dtype: float64
False labels (pleural_effusion):
113     0.0
129     0.0
165     0.0
304     0.0
315     0.0
       ... 
9808    0.0
9834    0.0
9844    0.0
9959    0.0
9983    0.0
Name: chexbert_pleural_effusion, Length: 138, dtype: float64


In [14]:
       
keep = usable_findings(st).values & label.notna().values
print(keep)

[False False False ... False  True False]


# 1. Make baseline

In [2]:
import os, sys
from pathlib import Path

PROJECT = Path.cwd().parent            # notebook folder → its parent = project root (where src/ and data/ are)
os.chdir(PROJECT)                      # so data/... paths in src/data.py work
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))   # so "import src" works

print("working dir:", Path.cwd())  

from src.data import binary_label, study_table, top_k_classes, usable_findings
from src.embeddings import EMBEDDINGS_DIR, embedding_path, load_embeddings

working dir: c:\Users\lucas\Desktop\4th_semester\02517_responsible_ai\code\mandatory\project3\responsible-ai-rexgradient-main


c:\Users\lucas\Desktop\4th_semester\02517_responsible_ai\code\mandatory\project3\responsible-ai-rexgradient-main\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from src.edits import sentences, find_sentence
print(sentences("Small right pleural effusion. Heart size is normal."))

['Small right pleural effusion.', 'Heart size is normal.']


In [ ]:
# Step 1: one frozen effusion classifier (70% train / 30% held-out, split by patient)

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# clear effusion cases with usable Findings (same 784 studies as Week 1)
studies = study_table() # Rturns all 10 k studies and w          
X_all = load_embeddings(embedding_path("findings"), studies) # returns the embeddings for all 10 k studies
label = binary_label(studies, "pleural_effusion") # creates a binary mask for the presence of pleural effusion in the studies and returns a series of 1s, 0s and NaNs
keep = usable_findings(studies).values & label.notna().values # usable findings returns a boolean mask of the studies that have usable findings ( more than 3 words). We keep only those are usable and have a label, excluding nan values
print(f"Keeping {keep.sum()} out of {len(keep)} studies")

df = studies[keep].reset_index(drop=True) # we apply the boolean mask to the studies     
X = X_all[keep] # we apply the same maske to the embeddings
y = label[keep].astype(int).values # we apply the mask to the labels and convert them to integers (1 for effusion, 0 for no effusion)
groups = df["patient_id"].values # we extract the patient ids for each study to ensure that we split by patient and not by study
print(f"{len(y)} studies: {y.sum()} positive, {(y == 0).sum()} negative")

# one split by patient: 70% train, 30% held-out
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
assert not set(groups[train_idx]) & set(groups[test_idx])     # no shared patients
print(f"train: {len(train_idx)} studies ({y[train_idx].mean():.0%} positive) | "
      f"held-out: {len(test_idx)} studies ({y[test_idx].mean():.0%} positive)")

# train the probe once and freeze it
probe = Pipeline([("scale", StandardScaler()),
                  ("clf", LogisticRegression(C=0.1, max_iter=3000))])
probe.fit(X[train_idx], y[train_idx])

# held-out data that all interventions will use
test_studies = df.iloc[test_idx].reset_index(drop=True)   # extracting the studies for the held-out set (texts, metadata)
X_test = X[test_idx] # extract the test embeddings for testing baseline model
y_test = y[test_idx] # extract the target from test
p_test = probe.predict_proba(X_test)[:, 1] # perform predictions

print(f"held-out AUROC: {roc_auc_score(y_test, p_test):.3f}")
print(f"held-out mean P(effusion): positives {p_test[y_test == 1].mean():.2f}, "
      f"negatives {p_test[y_test == 0].mean():.2f}")

784 studies: 646 positive, 138 negative
train: 554 studies (82% positive) | held-out: 230 studies (82% positive)
held-out AUROC: 0.911
held-out mean P(effusion): positives 0.93, negatives 0.36


# 2. Make edits

In [ ]:

from src.edits import sentences, find_sentence
#help(find_sentence)                                   

pos = test_studies[y_test == 1] # focusing on held-out reports WITH effusion                           
for text in pos["findings"].head(10): # iterating over the first 10 findings of the positive reports
    eff = [s for s in sentences(text) if "effusion" in s.lower()]
    print("-", eff)

print("\nshare of positive reports mentioning 'effusion':",
      pos["findings"].str.contains("effusion", case=False).mean().round(2))
print("share mentioning 'left' or 'right':",
      pos["findings"].str.contains(r"\b(left|right)\b", case=False).mean().round(2))

Help on function find_sentence in module src.edits:

find_sentence(text: str, word: str = 'effusion') -> int | None
    Index of the first sentence that contains `word` (case-insensitive), or None if there is none.

- []
- ['The patient has a small left pleural effusion with basilar atelectasis.']
- ['Bilateral pleural effusions and bibasilar atelectasis.']
- ['Some additional gradient veiling opacity in the left lung base and obscuration of the left hemidiaphragm which could reflect layering pleural effusion and/or basilar airspace disease.']
- ['Persistent small left pleural effusion.']
- ['Bibasilar atelectasis and bilateral pleural effusions are again seen without significant change.']
- ['There is bibasilar atelectasis with probable bilateral effusions.']
- ['There is a small right pleural effusion.']
- []
- ['Small bilateral pleural effusions are noted.']

share of positive reports mentioning 'effusion': 0.92
share mentioning 'left' or 'right': 0.85


C:\Users\lucas\AppData\Local\Temp\ipykernel_27992\489708509.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  pos["findings"].str.contains(r"\b(left|right)\b", case=False).mean().round(2))


In [6]:
test   = df.iloc[test_idx].reset_index(drop=True)   # held-out studies: text, patient, metadata
X_test = X[test_idx]                                 # their embeddings
y_test = y[test_idx]                                 # their labels

In [7]:
print(repr(test.loc[0, "findings"]))
print(test["findings"].isna().sum(), "missing texts")

'Lung volumes are low. Mild elevation of right hemidiaphragm. Ill-defined right infrahilar opacity is likely atelectasis. Heart at the upper limits normal in size, accentuated by technique. Questionable blunting of both costophrenic angles. No evidence of pulmonary edema. No pneumothorax. Surgical hardware about the right humerus, partially included. Gaseous distention of bowel in the upper abdomen, partially included.'
0 missing texts


- Negate
    - Apply to positives mentioning effusion 
    - Hypothesis is probability of effusion drops in the absence of positives 

- swap small <-> large

    - apply to the words "small" "large" specifically 
    - hypothesis doesnt change the prsence of effusion so P(effusion) should stay the same

- Insert Bibasilar Atelctasis
    - Apply yo cases where effusion is absent 
    - hypothesis bibasilar atelectasis is correlated with effusion. so potentially stay the same or rise

## 1. Negation

In [ ]:
from src.edits import sentences, find_sentence

texts = test["findings"].tolist()              # original held-out texts (row i <-> X_test[i])

neg_idx = []                                   # row positions in `test` that we edit
neg_texts = []                                 # the edited texts, same order as neg_idx

for i, t in enumerate(texts):
    if y_test[i] != 1:                         # only reports with effusion
        continue
    first = find_sentence(t, "effusion")       # index of first sentence with "effusion", or None
    if first is None:                          # effusion not mentioned in Findings -> can't negate
        continue
    sents = sentences(t)
    new_sents = []
    for j, s in enumerate(sents):
        if j == first:
            new_sents.append("No pleural effusion.")      # replace the first effusion sentence
        elif "effusion" in s.lower():
            continue                                       # drop any further effusion sentences
        else:
            new_sents.append(s)                            # keep everything else unchanged
    neg_idx.append(i)
    neg_texts.append(" ".join(new_sents))

print(f"positives in held-out: {(y_test == 1).sum()} | negation applied to: {len(neg_idx)}")

# read a few to check the edit did what we intended
for k in range(3):
    print("\nBEFORE:", texts[neg_idx[k]])
    print("AFTER: ", neg_texts[k])

positives in held-out: 189 | negation applied to: 173

BEFORE: The patient has a small left pleural effusion with basilar atelectasis. No pneumothorax is identified. Mild right basilar atelectasis is noted. The left lung is clear. Heart size is normal. Fractures are seen of the left sixth, seventh, eighth and tenth ribs. There may be a ninth rib fracture which is obscured by soft tissue structures.
AFTER:  No pleural effusion. No pneumothorax is identified. Mild right basilar atelectasis is noted. The left lung is clear. Heart size is normal. Fractures are seen of the left sixth, seventh, eighth and tenth ribs. There may be a ninth rib fracture which is obscured by soft tissue structures.

BEFORE: The support apparatus is stable given the difference in positioning. The heart remains enlarged. Persistent asymmetric right upper lobe airspace process. Bilateral pleural effusions and bibasilar atelectasis. No pneumothorax.
AFTER:  The support apparatus is stable given the difference in pos

In [ ]:
from src.embeddings import embed_texts, load_encoder

tokenizer, bert = load_encoder()                                   # same frozen BioClinicalBERT
neg_idx = np.array(neg_idx)

# embeddings: before = original (from Week 1), after = new (edited text)
X_before_neg = X_test[neg_idx]                                     # (173, 768)
X_after_neg = embed_texts(neg_texts, tokenizer, bert, verbose=False)   # (173, 768)

# probabilities from the frozen probe (no refitting)
p_before_neg = p_test[neg_idx]
p_after_neg = probe.predict_proba(X_after_neg)[:, 1]

# flip: did the decision (P >= 0.5) change?
flip_neg = (p_before_neg >= 0.5) != (p_after_neg >= 0.5)

# representation shift: cosine distance = 1 - cosine similarity
cos_sim = (X_before_neg * X_after_neg).sum(1) / (
    np.linalg.norm(X_before_neg, axis=1) * np.linalg.norm(X_after_neg, axis=1))
cos_neg = 1 - cos_sim

print(f"Negation, n = {len(neg_idx)}")
print(f"mean P(effusion) before: {p_before_neg.mean():.3f}")
print(f"mean P(effusion) after:  {p_after_neg.mean():.3f}")
print(f"flip rate:               {flip_neg.mean():.3f}")
print(f"mean cosine shift:       {cos_neg.mean():.4f}")

Negation, n = 173
mean P(effusion) before: 0.941
mean P(effusion) after:  0.516
flip rate:               0.434
mean cosine shift:       0.0089
Negation, n = 173
mean P(effusion) before: 0.941
mean P(effusion) after:  0.516
flip rate:               0.434
mean cosine shift:       0.0089


## 2. Swap

In [ ]:
import re

size = {"small": "large", "large": "small"}
pattern = r"\b(?:small|large)\b"

size_idx = []
size_texts = []

for i, t in enumerate(texts):
    if y_test[i] != 1:                                   # only reports with effusion
        continue
    if not re.search(pattern, t, flags=re.I):            # needs a size word
        continue
    edited = re.sub(pattern,
                    lambda m: size[m.group(0).lower()].capitalize() if m.group(0)[0].isupper()
                              else size[m.group(0).lower()],
                    t, flags=re.I)                       # Small -> Large, small -> large, ...
    size_idx.append(i)
    size_texts.append(edited)

print(f"positives in held-out: {(y_test == 1).sum()} | size swap applied to: {len(size_idx)}")

for k in range(3):
    print("\nBEFORE:", texts[size_idx[k]])
    print("AFTER: ", size_texts[k])

positives in held-out: 189 | size swap applied to: 101

BEFORE: The patient has a small left pleural effusion with basilar atelectasis. No pneumothorax is identified. Mild right basilar atelectasis is noted. The left lung is clear. Heart size is normal. Fractures are seen of the left sixth, seventh, eighth and tenth ribs. There may be a ninth rib fracture which is obscured by soft tissue structures.
AFTER:  The patient has a large left pleural effusion with basilar atelectasis. No pneumothorax is identified. Mild right basilar atelectasis is noted. The left lung is clear. Heart size is normal. Fractures are seen of the left sixth, seventh, eighth and tenth ribs. There may be a ninth rib fracture which is obscured by soft tissue structures.

BEFORE: Interval removal of NG tube. Feeding tube is been straightened, its tip is below left hemidiaphragm. Endotracheal tube and right PICC line in stable position. Drainage catheter right upper quadrant is in stable position. Heart size normal. P

In [ ]:
# embed the size-swapped texts, score with the frozen probe, measure the change
size_idx = np.array(size_idx)

X_before_size = X_test[size_idx]
X_after_size = embed_texts(size_texts, tokenizer, bert, verbose=False)

p_before_size = p_test[size_idx]
p_after_size = probe.predict_proba(X_after_size)[:, 1]

flip_size = (p_before_size >= 0.5) != (p_after_size >= 0.5)

cos_sim = (X_before_size * X_after_size).sum(1) / (
    np.linalg.norm(X_before_size, axis=1) * np.linalg.norm(X_after_size, axis=1))
cos_size = 1 - cos_sim

print(f"Size swap, n = {len(size_idx)}")
print(f"mean P(effusion) before: {p_before_size.mean():.3f}")
print(f"mean P(effusion) after:  {p_after_size.mean():.3f}")
print(f"flip rate:               {flip_size.mean():.3f}")
print(f"mean cosine shift:       {cos_size.mean():.4f}")

Size swap, n = 101
mean P(effusion) before: 0.947
mean P(effusion) after:  0.946
flip rate:               0.000
mean cosine shift:       0.0003


## 3. Insert

In [ ]:
atel_idx = []
atel_texts = []

for i, t in enumerate(texts):
    if y_test[i] != 0:                                   # only reports without effusion
        continue
    atel_idx.append(i)
    atel_texts.append(t.rstrip() + " Bibasilar atelectasis.")

print(f"negatives in held-out: {(y_test == 0).sum()} | insertion applied to: {len(atel_idx)}")
print("already mention atelectasis:",
      sum("atelectasis" in texts[i].lower() for i in atel_idx))

for k in range(2):
    print("\nBEFORE:", texts[atel_idx[k]])
    print("AFTER: ", atel_texts[k])

negatives in held-out: 41 | insertion applied to: 41
already mention atelectasis: 8

BEFORE: There is patchy hazy airspace opacity in both lungs, left greater than right. Disease shows a peripheral and basilar predominance. No associated pleural effusion. The cardiopericardial silhouette is within normal limits for size. The visualized bony structures of the thorax show now acute abnormality.
AFTER:  There is patchy hazy airspace opacity in both lungs, left greater than right. Disease shows a peripheral and basilar predominance. No associated pleural effusion. The cardiopericardial silhouette is within normal limits for size. The visualized bony structures of the thorax show now acute abnormality. Bibasilar atelectasis.

BEFORE: Interval resolution of small bilateral pleural effusions. There is no significant pulmonary consolidation, pulmonary edema, nor pneumothorax. Stable mild cardiomegaly. No grossly displaced rib fracture is noted.
AFTER:  Interval resolution of small bilateral pl

In [ ]:
# embed, score with the frozen probe, measure the change
atel_idx = np.array(atel_idx)

X_before_atel = X_test[atel_idx]
X_after_atel = embed_texts(atel_texts, tokenizer, bert, verbose=False)

p_before_atel = p_test[atel_idx]
p_after_atel = probe.predict_proba(X_after_atel)[:, 1]

flip_atel = (p_before_atel >= 0.5) != (p_after_atel >= 0.5)

cos_sim = (X_before_atel * X_after_atel).sum(1) / (
    np.linalg.norm(X_before_atel, axis=1) * np.linalg.norm(X_after_atel, axis=1))
cos_atel = 1 - cos_sim

print(f"Insert atelectasis, n = {len(atel_idx)}")
print(f"mean P(effusion) before: {p_before_atel.mean():.3f}")
print(f"mean P(effusion) after:  {p_after_atel.mean():.3f}")
print(f"flip rate:               {flip_atel.mean():.3f}")
print(f"mean cosine shift:       {cos_atel.mean():.4f}")

Insert atelectasis, n = 41
mean P(effusion) before: 0.362
mean P(effusion) after:  0.544
flip rate:               0.171
mean cosine shift:       0.0038


# **Poster**

Introduction:

- **Setup**: One effusion probe is trained once on 70 % of the clear effusion cases and then frozen. The reamining 30 % reports are edited, reembedded with the same BioclinicalBERT. The probe is then use to score the new embeddings and compare against baseline.

- **The edits**: 
    - Three edits are applied to the test set
    - Negation: replace effusion senetences with "no pleural effusion". Hypothesis is this should decrease P(effusion) and potentially flip some cases
    - Size swaps: "small" is swapped with "large" and vice versa. Hypothesis: This probabilites should remain more or less the same, since the presence of effusion is not changed.
    - Insertion: Insert "Bibasilar atelectasis" into cases with no effusion. Hypothesis: This should increase probability of effusion since Bibasilar atelectasis is often mentioned together with effusion. The resuslts are presented in the table below.

**Interpretation**
- *Were the hypotheses confirmed?*
    - Ngeation: Partly confirmed. P(effusion) droppped on average ( 0.94 -> 0.52). The embeddings do react to "no effusion". But only 43 % of the reports flipped below 0.5. The low cosine shift indicates the pooling of the reports into one embedding is affected by more than just the presence of effusion in the description.  

    - Size swap: P stayed the same (0.95 -> 0.95) and no report flipped. The model treats size as a change in the description and not in the presence of effusion 

    - Insertion: P(effusion) increased ( 0.36 -> 0.54) with 17 % of the reports flipped to effusion. Atelectasis occurs with effusion frequently enough to have an impact that pushes the prediction towards effusion without expliciutly mentioning effusion itself.

